# OrbitGPT in two cells

A tiny GPT you train yourself, then chat with. **Cell 1** fetches the code,
**Cell 2** trains it and opens the chat. That is the whole procedure.

* free T4: ~5 minutes for the default `micro` model (4.8 M parameters)
* no Google Drive, no downloads of model weights, no API keys
* run cell 2 again later and it reuses the checkpoint instead of retraining

In [ ]:
# ===========================================================================
#  CELL 1 of 2 - setup.  Run it once per Colab session (~20 seconds).
# ===========================================================================
import os

if not os.path.isdir("orbit-gpt"):
    !git clone -q --depth 1 https://github.com/Orbitlol/orbit-gpt.git

%cd orbit-gpt
!pip install -q ddgs        # optional: web search. Delete this line to skip.

print("ready - now run the next cell")

In [ ]:
# ===========================================================================
#  CELL 2 of 2 - train, then chat.  This is the whole thing.
#
#  First run  : trains for a few minutes on a free T4, then opens the chat.
#  Later runs : finds the saved model and goes straight to chatting.
#
#  Nothing is written to Google Drive - checkpoints live in /content.
#  Optional flags:
#    %run colab/orbit_gpt_colab.py --preset mini --max-steps 3000
#    %run colab/orbit_gpt_colab.py --retrain        # throw away the old model
#    %run colab/orbit_gpt_colab.py --no-search      # never touch the network
#    %run colab/orbit_gpt_colab.py --skip-pretrain  # one stage, weaker replies
# ===========================================================================
%run colab/orbit_gpt_colab.py

### What cell 2 actually does

1. **Stage 1 - pre-training** on generated prose so the model learns how
   English sentences are built. This is the step that stops the replies
   being word salad.
2. **Stage 2 - supervised fine-tuning (SFT)** on ~48k `User:` / `Assistant:`
   exchanges, starting from the stage-1 weights, so it answers in the chat
   format.
3. **Chat.** Optionally with web search, which is off unless the question
   needs fresh facts.

To take the model home, export it from the same notebook:

```
!python -m orbit_gpt.export checkpoints/orbit --out exports/orbit.onnx
!python orbit_app.py --model exports/orbit-int8.onnx --no-gui
```

`exports/orbit-int8.onnx` is about 1.3 MB and runs on a laptop CPU with
`pip install onnxruntime` - no PyTorch needed.